# Lesson 8: Choose units

**You will learn:** how to choose units with a functional localizer, as an fMRI study localizes a region, and compare their lesion with matched random units.
**Requires:** CPU only, about a minute.

**The question:** does silencing the most animal-selective units of the last layer hurt recognition of animals more than recognition of vehicles, and more than silencing random units? The localizer uses the training half of the images; the test uses the other half.

## Setup

Imports and settings.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. The **extractor** prepares images and records layers; the **region map** says which layer stands for which brain area (a hypothesis, not a fact). `behavioral_readout_layer` names the layer a readout answers from ([Lesson 6](06_behavior.ipynb)). `intervention` lets an experiment change the network during a run ([Lesson 7](07_perturb_a_model.ipynb)).

In [2]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',       # answers are read out from this layer
        intervention=build_pytorch_ablation_fn(network))   # carries out requested changes

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(subject.out_channels)                          # what this subject can produce

['behavior',
 'behavior:choice',
 'behavior:probabilities',
 'behavior:rating',
 'behavior:rt',
 'neural:IT',
 'neural:V1',
 'neural:V4',
 'perturbation']

## The experiment

### Stimuli

In [3]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [4]:
# Half the images (6 per category) fit the readout; the other half test it.
position = stimuli.groupby('category').cumcount()          # 0..11 within each category
train = StimulusSet(stimuli[position < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[position >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

### Task

The subject answers "animal" or "vehicle" for each test image. The readout is fitted once, on the intact network (`readout_fit='intact'`), and kept for every condition.

In [5]:
from brainscore_core import TaskContext
from brainscore_core.responses import Choice

task = TaskContext(response=Choice(['animal', 'vehicle']), fitting_stimuli=train,
                   prefer_path='readout', readout_fit='intact')

### Conditions

Three conditions: the intact network, the animal-selective units silenced, and the same number of random units silenced.

- `FunctionalSelection` records a region on localizer images, scores each unit by how much more it responds to one category than another (Cohen's d), and keeps the top units. It uses the training images only.
- `RandomSelection` draws the same number of units at random from the same population: the control every localized lesion needs.

In [6]:
from brainscore_core.selection import FunctionalSelection, RandomSelection

K = 2000          # of 86,528 units in features.12 (512 channels x 13 x 13 positions)
# Compare animal with vehicle images (the 'label' column) and keep the K most animal-selective units.
animal_units = FunctionalSelection('IT', train, contrast=(['animal'], ['vehicle']),
                                   contrast_column='label', n_units=K).resolve(subject)

# Draw the random units from the same population, addressed the same way: units counted over the
# flattened layer (axis and expected_shape copied), so an index means the same unit in both selections.
population = animal_units.metadata['unit_population']
random_units = RandomSelection(animal_units.layer, n_units=K, n_total=len(population),
                               population=population, seed=0, axis=animal_units.axis,
                               expected_shape=animal_units.expected_shape).resolve(subject)
subject.reset()                                 # the localizer run leaves nothing behind
print(animal_units.layer, len(animal_units.indices), 'animal-selective units;',
      len(random_units.indices), 'random units')

features.12 2000 animal-selective units; 2000 random units


In [7]:
from brainscore_core import Perturbation, StateChange

def silence(selection):
    """A request to set these units to zero."""
    return StateChange(kind='ablation', target=selection, perturbation=Perturbation(kind='zero'))

conditions = {'intact': None, 'animal-selective silenced': animal_units,
              'random silenced': random_units}

## Run

`answers()` shows the test images once and returns the subject's choices. A silenced condition applies its change, collects the answers, then undoes the change.

In [8]:
from brainscore_core.streaming import BufferedSession
from brainscore_core.streaming_helpers import apply_state_change

def answers():
    """Present the test images once; return the subject's choices as an assembly."""
    session = BufferedSession(stimulus_set=test, task_context=task,
                              requested_output_channels=['behavior:choice'])
    subject.participate(session)
    return session.collect('behavior:choice')

choices = {}                                    # condition -> choice assembly
for name, selection in conditions.items():
    if selection is None:
        choices[name] = answers()
        continue
    applied = apply_state_change(subject, silence(selection))             # silence the units
    choices[name] = answers()
    apply_state_change(subject, StateChange(kind='reset', handle_id=applied.handle_id))  # undo
subject.reset()

## Read results back

Each answer is an assembly with one row per test image and one column per choice; the chosen label is the column with the 1.

In [9]:
intact = choices['intact']
pd.DataFrame(intact.values, index=intact['stimulus_id'].values, columns=intact['choice'].values).head(3)

,animal,vehicle
airplane_06,0.0,1.0
airplane_07,0.0,1.0
airplane_08,1.0,0.0


## Analysis

In [10]:
LABELS = np.array(['animal', 'vehicle'])
truth = np.asarray(test['label'])

def accuracy_by_kind(choice_values):
    """Fraction correct among animal images and among vehicle images."""
    chosen = LABELS[np.asarray(choice_values).argmax(axis=1)]
    return {f'{kind} accuracy': float(np.mean(chosen[truth == kind] == kind)) for kind in LABELS}

## Results

In [11]:
results = pd.DataFrame({name: accuracy_by_kind(assembly.values) for name, assembly in choices.items()}).T
results.round(3)

,animal accuracy,vehicle accuracy
intact,0.889,0.750
animal-selective silenced,0.861,0.792
random silenced,0.889,0.750


A localized lesion is informative only next to the random one: if both lower accuracy alike, the selected units were not special. If the animal-selective lesion lowers animal accuracy more than vehicle accuracy, and more than the random lesion does, the localizer found units the readout relies on. Here silencing 2,000 animal-selective units lowered animal accuracy from 0.889 to 0.861 while vehicle accuracy rose from 0.750 to 0.792; the random lesion changed neither. With one random draw this is a first look; a study would repeat the random control with several seeds.

## Summary

- `FunctionalSelection` localizes units by contrast; `RandomSelection` matches them.
- Localize on one half of the images, test on the other.

**Next:** [Lesson 9](09_closed_loop.ipynb) runs a subject in a closed loop.